# Module 6.3: Transfer Learning & Ultralytics YOLO on Conveyor Inspection
### 🇵🇱 Transfer Learning i Ultralytics YOLO w Inspekcji Linii Przenośnikowej

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m06_l03_transfer_learning_yolo_conveyor_inspection.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Leverage transfer learning and single-stage YOLO object detection to locate and classify packaging and machining defects on moving conveyor lines.  
> 🇵🇱 **Streszczenie:** Wykorzystaj transfer learning i detekcję obiektów YOLO do lokalizowania i klasyfikacji wad wyrobów na ruchomych liniach przenośnikowych.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Manufacturing Data Scarcity**: Real defects occur rarely ($< 50$ cases/month). Training from scratch overfits; Transfer Learning fine-tunes pretrained backbones.
2. **Single-Stage YOLO Detection**: Predicts bounding box coordinates $[x, y, w, h]$ and class scores in a single forward pass, achieving $> 60	ext{ FPS}$ on conveyor belts.
3. **Intersection over Union (IoU)**:
   $$	ext{IoU} = rac{	ext{Area of Overlap}}{	ext{Area of Union}}$$
   Industry standard metric requiring $	ext{IoU} \ge 0.50$ for valid True Positive detection.


In [ ]:
# 1. Bounding Box Geometry & IoU Vectorized Calculation
def calculate_iou(box1, box2):
    # Format: [x1, y1, x2, y2]
    xa = max(box1[0], box2[0])
    ya = max(box1[1], box2[1])
    xb = min(box1[2], box2[2])
    yb = min(box1[3], box2[3])
    
    inter_area = max(0.0, xb - xa) * max(0.0, yb - ya)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

# Conveyor workpiece defect: Ground Truth annotation vs YOLO candidate proposal
gt_box   = [120.0, 90.0, 240.0, 210.0]  # [x1, y1, x2, y2]
yolo_box = [130.0, 85.0, 255.0, 205.0]

iou_score = calculate_iou(gt_box, yolo_box)
print(f"Calculated Proposal IoU: {iou_score:.3f} ({(iou_score*100):.1f}%)")


In [ ]:
# 2. Visualizing Defect Bounding Boxes on Conveyor Frame
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(8, 5))
# Conveyor belt black rubber background
conveyor = np.random.normal(0.15, 0.02, (300, 400))
ax.imshow(conveyor, cmap='gray', vmin=0, vmax=1)

# Draw ground truth bounding box (Green)
rect_gt = patches.Rectangle((gt_box[0], gt_box[1]), gt_box[2]-gt_box[0], gt_box[3]-gt_box[1],
                            linewidth=2.5, edgecolor='#10b981', facecolor='none', label='Human Annotated Ground Truth')
ax.add_patch(rect_gt)

# Draw YOLO predicted bounding box (Amber)
rect_yolo = patches.Rectangle((yolo_box[0], yolo_box[1]), yolo_box[2]-yolo_box[0], yolo_box[3]-yolo_box[1],
                              linewidth=2.5, edgecolor='#f59e0b', linestyle='--', facecolor='none', 
                              label=f'YOLO Proposal (IoU = {iou_score:.2f})')
ax.add_patch(rect_yolo)

plt.title("High-Speed Conveyor Belt Defect Localization (IoU Evaluation)", fontsize=13, fontweight='bold')
plt.legend(loc='lower right')
plt.xlim(0, 400)
plt.ylim(300, 0)
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Configure pretrained vision backbone weights using transfer learning *(🇵🇱 Skonfiguruj wagi wytrenowanego modelu wizyjnego za pomocą uczenia transferowego)*
- [ ] **Task 2:** Format defect bounding boxes in normalized coordinates [x_center, y_center, w, h] *(🇵🇱 Sformatuj ramki ograniczające defektów w znormalizowanych współrzędnych [x, y, w, h])*
- [ ] **Task 3:** Calculate Intersection over Union (IoU) and evaluate mAP@0.50 detection metrics *(🇵🇱 Oblicz metrykę Intersection over Union (IoU) i oceń jakość detekcji mAP@0.50)*


In [ ]:
# Exercise Challenge:
# Verify that the YOLO proposal meets the industrial acceptance threshold (IoU >= 0.50)
passes_qc = iou_score >= 0.50
print(f"Inspection Result: {'VALID DETECTION (True Positive)' if passes_qc else 'REJECT'}")
assert passes_qc, "YOLO bounding box failed to meet 50% IoU quality standard."
print("✓ Conveyor inspection evaluation verified.")
